
# Intelligent Course Recommendation System (RAG)
This notebook implements an advanced Retrieval-Augmented Generation (RAG) system for course recommendations. 

**Key Improvements in this version:**
1. **Object-Oriented Design**: Core logic is encapsulated in a `CourseRAG` class.
2. **Advanced Retrieval**: Implements a two-stage retrieval pipeline (Bi-Encoder for fast semantic search + Cross-Encoder for high-precision re-ranking).
3. **Data Cleaning**: Improved encoding handling to fix text corruption (e.g., garbled time formats).
4. **Structured LLM Output**: Uses Groq's JSON mode for reliable, parsable recommendations.
5. **Interactive UI**: Includes `ipywidgets` for a seamless search experience.

In [3]:
import os
import csv
import json
import math
from pathlib import Path
from typing import Any, Dict, List, Tuple

import pandas as pd
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer, CrossEncoder
import ipywidgets as widgets
from IPython.display import display, clear_output
from dotenv import load_dotenv

# Load environment variables (ensure you have a .env file with GROQ_API_KEY)
load_dotenv()

# Optional: Groq LLM integration
USE_GROQ = bool(os.environ.get("GROQ_API_KEY"))
if USE_GROQ:
    from groq import Groq

# Constants
DATA_DIR = Path(".")
COURSES_CSV = DATA_DIR / "courses_full_dataset_combined_courses.csv"
REVIEWS_CSV = DATA_DIR / "course_review.csv"
DB_PATH = DATA_DIR / "chroma_db"

if not COURSES_CSV.exists():
    COURSES_CSV = DATA_DIR / "cmu_labeled_llm_final.csv"

print(f"Courses CSV path: {COURSES_CSV} - Exists: {COURSES_CSV.exists()}")
print(f"Reviews CSV path: {REVIEWS_CSV} - Exists: {REVIEWS_CSV.exists()}")

Courses CSV path: courses_full_dataset_combined_courses.csv - Exists: True
Reviews CSV path: course_review.csv - Exists: True


## 1. Data Cleaning and Preprocessing
Robust data parsers to handle course schedules, clean text encoding issues, and aggregate review data.

In [4]:
def parse_days(weekday_str: str) -> List[str]:
    if not weekday_str or weekday_str == "TBA":
        return []
    valid_chars = {'M', 'T', 'W', 'R', 'F', 'S', 'U'}
    return [char for char in str(weekday_str).upper() if char in valid_chars]

def normalize_time_str(value: str) -> str:
    if value is None:
        return ""
    clean = str(value).encode("ascii", "ignore").decode("ascii").strip().upper()
    clean = " ".join(clean.replace(".", "").split())
    if clean in {"", "TBA", "N/A", "NA"}:
        return ""
    parts = clean.split()
    if len(parts) != 2:
        return clean
    time_part, period = parts
    if ":" not in time_part:
        time_part = f"{time_part}:00"
    return f"{time_part} {period}"

def parse_time_slots(start_str: str, end_str: str) -> List[str]:
    start_str = normalize_time_str(start_str)
    end_str = normalize_time_str(end_str)
    if not start_str or not end_str:
        return []

    def to_mins(t_str):
        try:
            time_part, period = t_str.split()
            h, m = map(int, time_part.split(":"))
            if period == "PM" and h != 12:
                h += 12
            if period == "AM" and h == 12:
                h = 0
            return h * 60 + m
        except Exception:
            return -1

    start_mins, end_mins = to_mins(start_str), to_mins(end_str)
    if start_mins == -1 or end_mins == -1 or start_mins >= end_mins:
        return []

    slots = []
    current_mins = start_mins
    while current_mins < end_mins:
        h, m = current_mins // 60, current_mins % 60
        period = "PM" if h >= 12 else "AM"
        display_h = h - 12 if h > 12 else (12 if h == 0 else h)
        slots.append(f"{display_h}:{m:02d} {period}")
        current_mins += 30
    return slots

def aggregate_reviews(reviews_path: Path) -> Dict[str, Dict[str, Any]]:
    if not reviews_path.exists():
        return {}

    reviews_df = pd.read_csv(reviews_path, encoding="utf-8", on_bad_lines="skip")
    if reviews_df.empty or "CourseID" not in reviews_df.columns:
        return {}

    for col in ["OverallRating", "WorkloadHours", "UtilityRating", "InterestRating"]:
        if col in reviews_df.columns:
            reviews_df[col] = pd.to_numeric(reviews_df[col], errors="coerce")

    grouped_reviews = {}
    for course_id, group in reviews_df.groupby("CourseID"):
        comments = []
        if "Comment" in group.columns:
            comments = [
                str(comment).strip().replace("\n", " ")
                for comment in group["Comment"].dropna().tolist()
                if str(comment).strip()
            ]

        grouped_reviews[str(course_id)] = {
            "review_count": int(len(group)),
            "avg_rating": round(group["OverallRating"].dropna().mean(), 2) if "OverallRating" in group.columns and not group["OverallRating"].dropna().empty else None,
            "avg_workload_hours": round(group["WorkloadHours"].dropna().mean(), 2) if "WorkloadHours" in group.columns and not group["WorkloadHours"].dropna().empty else None,
            "avg_utility": round(group["UtilityRating"].dropna().mean(), 2) if "UtilityRating" in group.columns and not group["UtilityRating"].dropna().empty else None,
            "avg_interest": round(group["InterestRating"].dropna().mean(), 2) if "InterestRating" in group.columns and not group["InterestRating"].dropna().empty else None,
            "review_snippets": comments[:3],
        }
    return grouped_reviews

def load_and_clean_data(courses_path: Path, reviews_path: Path):
    courses = []
    review_lookup = aggregate_reviews(reviews_path)

    with open(courses_path, "r", encoding="utf-8", errors="replace") as f:
        reader = csv.DictReader(f)
        for row in reader:
            weekday = row.get("weekday", "")
            start = row.get("start", "")
            end = row.get("end", "")
            row["_days"] = parse_days(weekday)
            row["_times"] = parse_time_slots(start, end)

            clean_start = normalize_time_str(start)
            clean_end = normalize_time_str(end)
            row["_meetingTime"] = f"{weekday} {clean_start}-{clean_end}" if weekday and clean_start else "TBA"

            skills_str = str(row.get("skills", "")).replace("[", "").replace("]", "").replace("'", "")
            row["skills"] = [s.strip() for s in skills_str.split(",") if s.strip()]

            review_summary = review_lookup.get(str(row.get("course_id", "")).strip(), {})
            row["review_count"] = review_summary.get("review_count", 0)
            row["avg_rating"] = review_summary.get("avg_rating")
            row["avg_workload_hours"] = review_summary.get("avg_workload_hours")
            row["avg_utility"] = review_summary.get("avg_utility")
            row["avg_interest"] = review_summary.get("avg_interest")
            row["review_snippets"] = review_summary.get("review_snippets", [])
            courses.append(row)

    print(f"Loaded {len(courses)} courses.")
    print(f"Aggregated reviews for {len(review_lookup)} unique courses.")
    return courses

# Load Data
COURSES = load_and_clean_data(COURSES_CSV, REVIEWS_CSV)
COURSES_BY_ID = {
    str(c["course_id"]): c
    for c in COURSES
    if str(c.get("course_id", "")).strip()
}

Loaded 8450 courses.
Aggregated reviews for 3145 unique courses.


## 2. The CourseRAG Engine
This class handles vectorization, the two-stage retrieval process (Bi-encoder + Cross-encoder), and structured LLM generation.

In [5]:
class CourseRAG:
    def __init__(self, db_path: str, use_groq: bool = False):
        self.db_path = Path(db_path)
        self.db_path.mkdir(exist_ok=True)

        self.client = chromadb.PersistentClient(
            path=str(self.db_path),
            settings=Settings(anonymized_telemetry=False, allow_reset=True)
        )
        self.collection = self.client.get_or_create_collection(
            name="courses_v3", metadata={"hnsw:space": "cosine"}
        )

        print("Loading Bi-Encoder...")
        self.bi_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        print("Loading Cross-Encoder for Re-ranking...")
        self.cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

        self.use_groq = use_groq
        if self.use_groq:
            self.llm_client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
            self.llm_model = "llama-3.3-70b-versatile"

    def _create_document(self, course: Dict) -> str:
        parts = []
        if course.get("course_id"):
            parts.append(f"Course: {course['course_id']}")
        if course.get("course_name"):
            parts.append(f"Title: {course['course_name']}")

        desc = course.get("description_clean") or course.get("description", "")
        if desc:
            parts.append(f"Description: {desc}")
        if course.get("industry"):
            parts.append(f"Industry: {course['industry']}")
        if course.get("level"):
            parts.append(f"Level: {course['level']}")
        if course.get("skills"):
            parts.append(f"Skills: {', '.join(course['skills'])}")
        if course.get("prerequisites"):
            parts.append(f"Prerequisites: {course['prerequisites']}")
        if course.get("_meetingTime") and course.get("_meetingTime") != "TBA":
            parts.append(f"Meeting time: {course['_meetingTime']}")

        review_count = course.get("review_count", 0)
        if review_count:
            parts.append(f"Review count: {review_count}")
        if course.get("avg_rating") is not None:
            parts.append(f"Average rating: {course['avg_rating']}/5")
        if course.get("avg_workload_hours") is not None:
            parts.append(f"Average workload hours: {course['avg_workload_hours']} per week")
        if course.get("avg_utility") is not None:
            parts.append(f"Average utility rating: {course['avg_utility']}/5")
        if course.get("review_snippets"):
            parts.append(f"Student feedback: {' || '.join(course['review_snippets'])}")

        return " | ".join(parts)

    def _matches_schedule(self, course: Dict, schedule_filter: Dict[str, set]) -> bool:
        if not schedule_filter:
            return True

        course_days = course.get("_days", [])
        course_times = course.get("_times", [])
        if not course_days or not course_times:
            return False

        for day in course_days:
            allowed_slots = schedule_filter.get(day)
            if not allowed_slots or any(slot not in allowed_slots for slot in course_times):
                return False
        return True

    def ingest_data(self, courses: List[Dict]):
        if self.collection.count() > 0:
            print(f"Collection already populated with {self.collection.count()} items. Skipping ingest.")
            return

        documents, metadatas, ids = [], [], []
        seen_ids = set()

        for idx, row in enumerate(courses):
            course_id = str(row.get("course_id", f"idx_{idx}"))
            if course_id in seen_ids:
                continue

            seen_ids.add(course_id)
            documents.append(self._create_document(row))
            ids.append(course_id)
            metadatas.append({
                "course_name": row.get("course_name", ""),
                "industry": row.get("industry", ""),
                "level": row.get("level", ""),
            })

        print(f"Embedding {len(documents)} unique documents...")
        embeddings = self.bi_encoder.encode(documents, show_progress_bar=True).tolist()

        batch_size = 5000
        for b in range(math.ceil(len(ids) / batch_size)):
            s, e = b * batch_size, (b + 1) * batch_size
            self.collection.upsert(
                ids=ids[s:e],
                embeddings=embeddings[s:e],
                documents=documents[s:e],
                metadatas=metadatas[s:e]
            )
        print("Ingestion complete.")

    def search(self, query: str, top_k: int = 10, schedule_filter: Dict = None) -> List[Tuple[float, Dict]]:
        initial_k = max(top_k * 8, 20)
        results = self.collection.query(query_texts=[query], n_results=initial_k)

        candidates = []
        for i, cid in enumerate(results["ids"][0]):
            course = COURSES_BY_ID.get(cid)
            if not course:
                continue
            if schedule_filter and not self._matches_schedule(course, schedule_filter):
                continue
            candidates.append((results["documents"][0][i], course))

        if not candidates:
            return []

        pairs = [[query, doc] for doc, _ in candidates]
        cross_scores = self.cross_encoder.predict(pairs)

        scored_results = []
        for i, score in enumerate(cross_scores):
            scored_results.append((float(score), candidates[i][1]))

        scored_results.sort(key=lambda x: x[0], reverse=True)
        return scored_results[:top_k]

    def generate_json_advice(self, query: str, top_k: int = 5, schedule_filter: Dict = None) -> Dict:
        if not self.use_groq:
            return {"error": "Groq API not configured."}

        hits = self.search(query, top_k=top_k, schedule_filter=schedule_filter)
        if not hits:
            return {
                "recommended_courses": [],
                "explanation": "No matching courses found for the current goal and schedule constraints.",
                "reason_per_course": []
            }

        context_blocks = []
        for score, course in hits:
            desc = (course.get("description_clean") or course.get("description") or "")[:300]
            review_summary = f"rating={course.get('avg_rating')}, workload={course.get('avg_workload_hours')}, reviews={course.get('review_count', 0)}"
            context_blocks.append(
                f"ID: {course.get('course_id')} | Title: {course.get('course_name')} | Level: {course.get('level')} | Time: {course.get('_meetingTime')} | {review_summary} | Desc: {desc}"
            )
        context_str = "\n".join(context_blocks)

        prompt = f"""You are an academic advisor. Based strictly on the provided context, recommend courses for the student.
Return valid JSON with exactly these keys:
- recommended_courses: a list of course IDs.
- explanation: a concise overall recommendation summary.
- reason_per_course: a list of objects with keys course_id, rationale, and confidence.

If the context is insufficient, return an empty recommended_courses list instead of inventing courses.

Query: {query}
Context:
{context_str}
"""
        response = self.llm_client.chat.completions.create(
            model=self.llm_model,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
            temperature=0.2,
            max_tokens=700
        )
        return json.loads(response.choices[0].message.content)

# Initialize System
rag_system = CourseRAG(db_path=str(DB_PATH), use_groq=USE_GROQ)
rag_system.ingest_data(COURSES)

Loading Bi-Encoder...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Embedding 3145 unique documents...


Batches:   0%|          | 0/99 [00:00<?, ?it/s]

Ingestion complete.


## 3. Interactive Search UI
Use the widgets below to test the RAG system directly within the notebook.

In [ ]:
# Create UI Elements
query_input = widgets.Text(
    value='I want to learn about product management and prototyping',
    placeholder='Type your learning goals...',
    description='Goal:',
    layout=widgets.Layout(width='80%')
)

day_selector = widgets.SelectMultiple(
    options=[('Monday', 'M'), ('Tuesday', 'T'), ('Wednesday', 'W'), ('Thursday', 'R'), ('Friday', 'F')],
    value=(),
    description='Days:',
    layout=widgets.Layout(width='300px', height='140px')
)

time_window_input = widgets.Text(
    value='',
    placeholder='e.g. 9:00 AM-12:00 PM',
    description='Free Time:',
    layout=widgets.Layout(width='320px')
)

search_btn = widgets.Button(
    description='Search & Analyze',
    button_style='primary',
    icon='search'
)

output_area = widgets.Output()

def build_schedule_filter(selected_days, time_window: str):
    if not selected_days:
        return None

    clean_window = (time_window or '').strip()
    if not clean_window:
        return None
    if '-' not in clean_window:
        raise ValueError("Time window should look like 9:00 AM-12:00 PM")

    start_str, end_str = [part.strip() for part in clean_window.split('-', 1)]
    slots = parse_time_slots(start_str, end_str)
    if not slots:
        raise ValueError("Could not parse the selected time window.")
    return {day: set(slots) for day in selected_days}

def on_search_clicked(b):
    with output_area:
        clear_output()
        print("Searching and re-ranking courses...")
        query = query_input.value.strip()
        if not query:
            print("Please enter a learning goal.")
            return

        try:
            schedule_filter = build_schedule_filter(day_selector.value, time_window_input.value)
        except ValueError as exc:
            print(f"Schedule filter error: {exc}")
            return

        if schedule_filter:
            print(f"Applying schedule filter for {', '.join(day_selector.value)} during {time_window_input.value}.")

        hits = rag_system.search(query, top_k=5, schedule_filter=schedule_filter)
        if not hits:
            print("No matching courses found under the current filters.")
            return

        print("\n--- Top Retrieved Courses (Cross-Encoder Re-ranked) ---")
        for score, course in hits:
            cid = course.get("course_id", "N/A")
            name = course.get("course_name", "Unknown")
            time = course.get("_meetingTime", "TBA")
            rating = course.get("avg_rating")
            workload = course.get("avg_workload_hours")
            reviews = course.get("review_count", 0)
            print(f"[{score:.2f}] {cid} - {name} ({time})")
            print(f"    rating={rating}, workload_hours={workload}, reviews={reviews}")

        if USE_GROQ:
            print("\nAI Advice (JSON Mode):")
            try:
                advice_json = rag_system.generate_json_advice(query, top_k=5, schedule_filter=schedule_filter)
                print(json.dumps(advice_json, indent=2))
            except Exception as e:
                print(f"Error generating advice: {e}")

search_btn.on_click(on_search_clicked)

# Display UI
display(widgets.VBox([
    widgets.HTML("<h3>Find Your Courses</h3>"),
    query_input,
    widgets.HBox([day_selector, time_window_input]),
    widgets.HTML("<small>Select weekdays and enter a free-time window if you want schedule-aware recommendations.</small>"),
    search_btn,
    output_area
]))